# Fase 17B.1 — Partições não-IID dos cinco clientes

Correção metodológica obrigatória antes dos treinamentos. Mantém os splits científicos aprovados na Fase 17B e congela exatamente cinco clientes não-IID usando somente o conjunto de treino. Nenhuma amostra de validação ou teste pode entrar nos clientes. No CheXchoNet, um paciente pertence a apenas um cliente.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil
import numpy as np
import pandas as pd

CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
PROJECT=Path('/content/drive/MyDrive/Mestrado_Criptografia')
CAMPAIGN=PROJECT/'OFFICIAL_CAMPAIGN_V2'/CAMPAIGN_ID
FREEZE=CAMPAIGN/'01_DATASET_FREEZE'
CONTROL=CAMPAIGN/'00_CAMPAIGN_CONTROL'
SEED=42; NUM_CLIENTS=5; DIRICHLET_ALPHA=0.5; MIN_CLIENT_SAMPLES=10
assert (CONTROL/'PHASE17B_MASTER_GATE.json').exists(), 'Execute a Fase 17B primeiro.'
print('Campanha:',CAMPAIGN)

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def load_split_npz(path):
    z=np.load(path,allow_pickle=True); out={}
    for k in z.files:
        lk=k.lower()
        if 'train' in lk and 'client' not in lk: out['train']=np.asarray(z[k]).reshape(-1).astype(np.int64)
        elif ('val' in lk or 'valid' in lk) and 'client' not in lk: out['validation']=np.asarray(z[k]).reshape(-1).astype(np.int64)
        elif 'test' in lk and 'client' not in lk: out['test']=np.asarray(z[k]).reshape(-1).astype(np.int64)
    return out

def dirichlet_partition(indices, labels, seed=42, alpha=0.5, n_clients=5, min_size=10, groups=None):
    indices=np.asarray(indices,dtype=np.int64); labels=np.asarray(labels)
    assert len(indices)==len(labels) and len(indices)>0
    rng=np.random.default_rng(seed)
    for attempt in range(1000):
        buckets=[[] for _ in range(n_clients)]
        if groups is None:
            for cls in sorted(np.unique(labels).tolist()):
                cls_idx=indices[labels==cls].copy(); rng.shuffle(cls_idx)
                props=rng.dirichlet(np.full(n_clients,alpha)); cuts=(np.cumsum(props)[:-1]*len(cls_idx)).astype(int)
                for c,part in enumerate(np.split(cls_idx,cuts)): buckets[c].extend(part.tolist())
        else:
            groups=np.asarray(groups); frame=pd.DataFrame({'idx':indices,'y':labels,'g':groups})
            group_label=frame.groupby('g')['y'].agg(lambda s:int(float(s.mean())>=0.5))
            for cls in sorted(group_label.unique().tolist()):
                gs=group_label[group_label==cls].index.to_numpy().copy(); rng.shuffle(gs)
                props=rng.dirichlet(np.full(n_clients,alpha)); cuts=(np.cumsum(props)[:-1]*len(gs)).astype(int)
                for c,gpart in enumerate(np.split(gs,cuts)):
                    if len(gpart): buckets[c].extend(frame[frame.g.isin(gpart)]['idx'].tolist())
        if min(map(len,buckets))>=min_size:
            return [np.array(sorted(b),dtype=np.int64) for b in buckets],attempt+1
    raise RuntimeError('Não foi possível obter cinco clientes com o mínimo de amostras.')

def validate_clients(clients, train_idx, labels_all, group_all=None):
    sets=[set(map(int,c)) for c in clients]; train=set(map(int,train_idx))
    disjoint=all(not sets[i]&sets[j] for i in range(len(sets)) for j in range(i+1,len(sets)))
    union=set().union(*sets) if sets else set()
    rates=[float(np.mean(labels_all[c])) for c in clients]
    group_disjoint=True
    if group_all is not None:
        gs=[set(np.asarray(group_all)[c].astype(str)) for c in clients]
        group_disjoint=all(not gs[i]&gs[j] for i in range(len(gs)) for j in range(i+1,len(gs)))
    return {'five_clients':len(clients)==5,'nonempty':all(len(c)>0 for c in clients),'disjoint':disjoint,'cover_train_exactly':union==train,'no_nontrain_samples':union<=train,'group_disjoint':group_disjoint,'client_counts':[len(c) for c in clients],'positive_rates':rates,'non_iid_rate_range':max(rates)-min(rates),'non_iid_demonstrated':max(rates)-min(rates)>=0.02}

def save_partition(dataset,clients,report):
    out=FREEZE/dataset/'SCIENTIFIC_FREEZE'; p=out/'official_client_partition_noniid_seed42.npz'
    np.savez_compressed(p,**{f'client_{i}':c for i,c in enumerate(clients)})
    report.update({'partition_path':str(p),'partition_sha256':sha256_file(p)})
    (out/'PHASE17B1_CLIENT_GATE.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    return report

In [ ]:
# PhysioNet — cada RecordID é uma unidade independente.
d='PHYSIONET_CHALLENGE_2012'; root=FREEZE/d/'SCIENTIFIC_FREEZE'
df=pd.read_csv(root/'physionet_challenge_2012_features.csv'); split=load_split_npz(root/'official_split_seed42.npz'); train=split['train']
y=pd.to_numeric(df['target']).to_numpy(dtype=int)
clients,attempts=dirichlet_partition(train,y[train],SEED,DIRICHLET_ALPHA,NUM_CLIENTS,MIN_CLIENT_SAMPLES)
checks=validate_clients(clients,train,y)
report={'dataset':d,'seed':SEED,'method':'label_dirichlet','alpha':DIRICHLET_ALPHA,'attempts':attempts,'anti_leakage_unit':'RecordID','checks':checks,'approved':all(v for k,v in checks.items() if k not in {'client_counts','positive_rates','non_iid_rate_range'})}
save_partition(d,clients,report); print(json.dumps(report,indent=2))

In [ ]:
# Dahl Rats — o split por animal permanece congelado; janelas do treino são distribuídas de forma não-IID.
d='DAHL_RATS'; root=FREEZE/d/'SCIENTIFIC_FREEZE'
df=pd.read_csv(root/'dahl_derived_features.csv'); split=load_split_npz(root/'official_split_seed42.npz'); train=split['train']
y=pd.to_numeric(df['target']).to_numpy(dtype=int)
clients,attempts=dirichlet_partition(train,y[train],SEED,DIRICHLET_ALPHA,NUM_CLIENTS,MIN_CLIENT_SAMPLES)
checks=validate_clients(clients,train,y)
report={'dataset':d,'seed':SEED,'method':'label_dirichlet_over_train_windows','alpha':DIRICHLET_ALPHA,'attempts':attempts,'split_anti_leakage_unit':'rat_id','checks':checks,'approved':all(v for k,v in checks.items() if k not in {'client_counts','positive_rates','non_iid_rate_range','group_disjoint'})}
save_partition(d,clients,report); print(json.dumps(report,indent=2))

In [ ]:
# CheXchoNet — particionamento não-IID agrupado por paciente.
d='CHEXCHONET'; root=FREEZE/d/'SCIENTIFIC_FREEZE'
z=np.load(root/'chexchonet_embeddings.npz',allow_pickle=True); X=z['embeddings']; y=np.asarray(z['targets']).astype(int)
raw=np.asarray(np.load(root/'official_split_original.npy',allow_pickle=True)).reshape(-1)
norm=lambda x:{'0':'train','1':'validation','2':'test','val':'validation','valid':'validation','dev':'validation'}.get(str(x).strip().lower(),str(x).strip().lower())
split_labels=np.array([norm(x) for x in raw]); train=np.where(split_labels=='train')[0].astype(np.int64)
pm=pd.read_csv(root/'chexchonet_patient_manifest.csv'); lower={c.lower():c for c in pm.columns}; pid=next(lower[k] for k in ['patient_id','patient','subject_id','subject','pid'] if k in lower)
assert len(pm)==len(y), f'Manifesto ({len(pm)}) não está alinhado aos embeddings ({len(y)}).'
groups=pm[pid].astype(str).to_numpy()
clients,attempts=dirichlet_partition(train,y[train],SEED,DIRICHLET_ALPHA,NUM_CLIENTS,MIN_CLIENT_SAMPLES,groups=groups[train])
checks=validate_clients(clients,train,y,groups)
report={'dataset':d,'seed':SEED,'method':'patient_grouped_label_dirichlet','alpha':DIRICHLET_ALPHA,'attempts':attempts,'anti_leakage_unit':'patient','patient_id_column':pid,'checks':checks,'approved':all(v for k,v in checks.items() if k not in {'client_counts','positive_rates','non_iid_rate_range'})}
save_partition(d,clients,report); print(json.dumps(report,indent=2))

In [ ]:
datasets=['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']; gates={}
for d in datasets: gates[d]=json.loads((FREEZE/d/'SCIENTIFIC_FREEZE'/'PHASE17B1_CLIENT_GATE.json').read_text())
all_ok=all(g['approved'] for g in gates.values())
master={'phase':'17B.1','campaign_id':CAMPAIGN_ID,'completed_at_utc':datetime.now(timezone.utc).isoformat(),'dataset_approvals':{d:g['approved'] for d,g in gates.items()},'five_clients_noniid_frozen':all_ok,'training_authorized':all_ok,'next_authorized_step':'FASE_17C_BASELINE_SMOKE_TESTS' if all_ok else 'REPAIR_CLIENT_PARTITION_GATES','partition_hashes':{d:g.get('partition_sha256') for d,g in gates.items()},'failed_checks':{d:[k for k,v in g['checks'].items() if isinstance(v,bool) and not v] for d,g in gates.items()}}
(CONTROL/'PHASE17B1_MASTER_GATE.json').write_text(json.dumps(master,indent=2,ensure_ascii=False),encoding='utf-8')
status=json.loads((CONTROL/'CAMPAIGN_STATUS.json').read_text()); status.update({'status':'PHASE17B1_COMPLETED' if all_ok else 'PHASE17B1_BLOCKED','phase17b1_training_authorized':all_ok}); (CONTROL/'CAMPAIGN_STATUS.json').write_text(json.dumps(status,indent=2,ensure_ascii=False))
print('='*100); print(json.dumps(master,indent=2,ensure_ascii=False)); print('='*100)

In [ ]:
export=Path('/content/PHASE17B1_EVIDENCE'); shutil.rmtree(export,ignore_errors=True); export.mkdir()
for name in ['CAMPAIGN_STATUS.json','PHASE17B_MASTER_GATE.json','PHASE17B1_MASTER_GATE.json']: shutil.copy2(CONTROL/name,export/name)
for d in ['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']:
    dst=export/d; dst.mkdir(); src=FREEZE/d/'SCIENTIFIC_FREEZE'; shutil.copy2(src/'PHASE17B1_CLIENT_GATE.json',dst/'PHASE17B1_CLIENT_GATE.json')
zip_path=shutil.make_archive('/content/PHASE17B1_EVIDENCE','zip','/content','PHASE17B1_EVIDENCE')
permanent=CONTROL/'EXPORTS'/'PHASE17B1_EVIDENCE.zip'; permanent.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(zip_path,permanent)
print('Salvo em:',permanent); files.download(str(permanent))